In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t
from scipy.stats import f
from scipy.stats import norm

In [2]:
class Regressor:
    def __init__(self, X: np.ndarray, y: np.ndarray, functions: list["function"], precise: float = 3, beta: float = 0.05) -> None:
        self.X = X
        self.y = y
        self.functions = functions
        self.beta = beta
        self.psi_matrix = np.array([[f(val) for f in self.functions] for val in self.X])
        self.F_inverted = self.findFInvertedMatrix()
        self.beta_estimations = self.findRegressionCoefficients()
        self.precise = precise

    def findFInvertedMatrix(self) -> np.ndarray:
        F_matrix = self.psi_matrix.T @ self.psi_matrix
        F_inverted_matrix = np.linalg.inv(F_matrix)

        return F_inverted_matrix

    def findRegressionCoefficients(self) -> np.ndarray:
        F_matrix = self.psi_matrix.T @ self.psi_matrix
        F_inverted_matrix = np.linalg.inv(F_matrix)
        beta_estimations = F_inverted_matrix @ self.psi_matrix.T @ self.y

        return beta_estimations
    
    def findErrors(self) -> np.ndarray:
        self.beta_estimations = self.findRegressionCoefficients()
        y_hat = self.psi_matrix @ self.beta_estimations
        return self.y - y_hat
    
    def findRSS(self) -> float:
        errors = self.findErrors()
        return errors.T @ errors
    
    def findTSS(self) -> float:
        y_avg = np.average(self.y)
        return np.sum((y_avg - self.y) ** 2)
    
    def findRSquare(self) -> float:
        TSS = self.findTSS()
        RSS = self.findRSS()
        return np.round((TSS - RSS) / TSS, self.precise)
    
    def checkCoefficientSignificance(self, ind: int) -> float:
        RSS = self.findRSS()
        F_inverted_i_i = self.F_inverted[ind, ind]
        beta_estimated = self.beta_estimations[ind]
        n = self.X.shape[0]
        p = len(self.functions)
        delta_estimation = abs(beta_estimated / np.sqrt(RSS * F_inverted_i_i / (n - p)))
        p_value = t.cdf(-delta_estimation, n - p) + t.sf(delta_estimation, n - p)
        return p_value
    
    def compareCoefficients(self, i: int, j: int) -> float:
        RSS = self.findRSS()
        F_inverted_i_i = self.F_inverted[i, i]
        F_inverted_j_j = self.F_inverted[j, j]
        beta_estimated_i = self.beta_estimations[i]
        beta_estimated_j = self.beta_estimations[j]
        n = self.X.shape[0]
        p = len(self.functions)
        delta_estimation = abs((beta_estimated_i - beta_estimated_j) / np.sqrt(RSS * (F_inverted_i_i + F_inverted_j_j) / (n - p)))
        p_value = t.cdf(-delta_estimation, n - p) + t.sf(delta_estimation, n - p)
        return p_value
    
    def findConfidenceInterval(self, point: np.ndarray) -> tuple[float]:
        prediction = self.getPrediction(point)
        n = self.X.shape[0]
        p = len(self.functions)
        RSS = self.findRSS()
        func_vals = np.array([func(point) for func in self.functions])
        val = np.sqrt(RSS * (1 + func_vals @ self.F_inverted @ func_vals.T) / (n - p))
        left_bound = (1 - self.beta) / 2
        right_bound = (1 + self.beta) / 2
        left_percentile = t.ppf(left_bound, n - p)
        right_percentile = t.ppf(right_bound, n - p)

        left_value = prediction - val * right_percentile
        right_value = prediction - val * left_percentile

        return (left_value, right_value)

    def getPrediction(self, point: np.ndarray) -> float:
        func_vals = np.array([func(point) for func in self.functions])
        prediction = self.beta_estimations.T @ func_vals
        return prediction
    
    def getPredictions(self, points: np.ndarray) -> np.ndarray:
        predictions = [self.getPrediction(point) for point in points]
        return predictions
    
    def findIndependance(self) -> float:
        errors = self.findErrors()
        n = errors.shape[0]
        inv_count = 0
        for i in range(n):
            for j in range(i + 1, n):
                if (errors[j] < errors[i]):
                    inv_count += 1
        
        middle_inv_count = n * (n - 1) // 4

        delta = abs((inv_count - middle_inv_count) / np.sqrt(n ** 3 / 36))
        p_value = norm.cdf(-delta) + norm.sf(delta)
        return p_value
    
    def printRegr(self) -> None:
        answ = ""
        for i in range(0, self.beta_estimations.shape[0]):
            coef = self.beta_estimations[i]
            answ += f" + {str(np.round(coef, self.precise))}*x_{i}"
        print("y =", answ)

In [3]:
y = np.array(
    [83, 85,
     84, 85, 85, 86, 86, 87,
     86, 87, 87, 87, 88, 88, 88, 88, 88, 89, 90,
     89, 90, 90, 91,
     90, 92]
)

indicator_funcs = np.eye(5)

X = np.array(
    [
        *np.repeat(indicator_funcs[0][np.newaxis,], 2, axis=0),
        *np.repeat(indicator_funcs[1][np.newaxis,], 6, axis=0),
        *np.repeat(indicator_funcs[2][np.newaxis,], 11, axis=0),
        *np.repeat(indicator_funcs[3][np.newaxis,], 4, axis=0),
        *np.repeat(indicator_funcs[4][np.newaxis,], 2, axis=0),
    ]
)

In [4]:
func0 = lambda x: x[0]
func1 = lambda x: x[1]
func2 = lambda x: x[2]
func3 = lambda x: x[3]
func4 = lambda x: x[4]
funcs = [func0, func1, func2, func3, func4]

regr = Regressor(X, y, funcs)

regr.printRegr()
for i in range(len(funcs)):
    print(f"p-value для значимости {i}-ого коэффициента: {regr.checkCoefficientSignificance(i)}")

print(f"R^2 = {regr.findRSquare()}")

y =  + 84.0*x_0 + 85.5*x_1 + 87.818*x_2 + 90.0*x_3 + 91.0*x_4
p-value для значимости 0-ого коэффициента: 2.433733308049601e-29
p-value для значимости 1-ого коэффициента: 2.923672428046543e-34
p-value для значимости 2-ого коэффициента: 4.001322125539143e-37
p-value для значимости 3-ого коэффициента: 6.0330973821268634e-33
p-value для значимости 4-ого коэффициента: 4.9207890401875334e-30
R^2 = 0.811


b) множественная проверка гипотез

In [5]:
arr = np.zeros(shape=(5, 5))
coeffs = []

for i in range(0, 5):
    for j in range(i + 1, 5):
        coeffs.append(regr.compareCoefficients(i, j))

ind_sorted = np.argsort(coeffs)

alpha = 0.05
alpha_adjusted = [alpha / i for i in range(10, 0, -1)]

rejected = []
for i in range(10):
    if coeffs[ind_sorted[i]] >= alpha_adjusted[i]:
        break
    rejected.append(ind_sorted[i])

rank_map = {orig_idx: i for i, orig_idx in enumerate(ind_sorted)}

index = 0
for i in range(0, 5):
    for j in range(i + 1, 5):
        rank = rank_map[index]
        if index in rejected:
            print(f"{i} и {j} коэффициенты не равны, p_value = {coeffs[index]:.4f}, alpha_corrected = {alpha_adjusted[rank]:.4f}")
        else:
            print(f"{i} и {j} коэффициенты РАВНЫ, p_value = {coeffs[index]:.4f}, alpha_corrected = {alpha_adjusted[rank]:.4f}")
        index += 1

0 и 1 коэффициенты РАВНЫ, p_value = 0.1031, alpha_corrected = 0.0250
0 и 2 коэффициенты не равны, p_value = 0.0002, alpha_corrected = 0.0083
0 и 3 коэффициенты не равны, p_value = 0.0000, alpha_corrected = 0.0063
0 и 4 коэффициенты не равны, p_value = 0.0000, alpha_corrected = 0.0050
1 и 2 коэффициенты не равны, p_value = 0.0004, alpha_corrected = 0.0100
1 и 3 коэффициенты не равны, p_value = 0.0000, alpha_corrected = 0.0056
1 и 4 коэффициенты не равны, p_value = 0.0000, alpha_corrected = 0.0071
2 и 3 коэффициенты не равны, p_value = 0.0024, alpha_corrected = 0.0167
2 и 4 коэффициенты не равны, p_value = 0.0010, alpha_corrected = 0.0125
3 и 4 коэффициенты РАВНЫ, p_value = 0.2958, alpha_corrected = 0.0500


In [6]:
from statsmodels.stats.multitest import multipletests
rejected, p_corrected, _, _ = multipletests(coeffs, alpha=0.05, method='holm')
index = 0
for i in range(0, 5):
    for j in range(i + 1, 5):
        if rejected[index]:
            print(f"Гипотеза ОТВЕРГНУТА: {i}-ый и {j}-ый коэффициенты НЕ равны (p_adj = {p_corrected[index]:.4f})")
        else:
            print(f"Гипотеза ПРИНЯТА: {i}-ый и {j}-ый коэффициенты равны")
        index += 1

Гипотеза ПРИНЯТА: 0-ый и 1-ый коэффициенты равны
Гипотеза ОТВЕРГНУТА: 0-ый и 2-ый коэффициенты НЕ равны (p_adj = 0.0010)
Гипотеза ОТВЕРГНУТА: 0-ый и 3-ый коэффициенты НЕ равны (p_adj = 0.0000)
Гипотеза ОТВЕРГНУТА: 0-ый и 4-ый коэффициенты НЕ равны (p_adj = 0.0000)
Гипотеза ОТВЕРГНУТА: 1-ый и 2-ый коэффициенты НЕ равны (p_adj = 0.0020)
Гипотеза ОТВЕРГНУТА: 1-ый и 3-ый коэффициенты НЕ равны (p_adj = 0.0000)
Гипотеза ОТВЕРГНУТА: 1-ый и 4-ый коэффициенты НЕ равны (p_adj = 0.0000)
Гипотеза ОТВЕРГНУТА: 2-ый и 3-ый коэффициенты НЕ равны (p_adj = 0.0072)
Гипотеза ОТВЕРГНУТА: 2-ый и 4-ый коэффициенты НЕ равны (p_adj = 0.0040)
Гипотеза ПРИНЯТА: 3-ый и 4-ый коэффициенты равны
